# Prompt 5 — Resource and Data Blocks
## Terraform Associate (004) Exam Study Guide

**Exam Objective 4 (HCL Configuration)**: Understand `resource` and `data` blocks — the two fundamental building blocks of every Terraform configuration.

---

**Topics covered in this notebook:**
1. The `resource` Block — Syntax and Structure
2. Resource Addressing
3. Resource Behaviors: Create, Read, Update, Destroy
4. Required vs Optional Arguments
5. The `data` Block — Reading Existing Infrastructure
6. Resource vs Data Source Comparison
7. When Data Sources Read: Plan vs Apply Timing
8. Lifecycle Meta-Arguments
9. Complete Examples
10. Exam-Style Questions (3)
11. Real-World Scenarios (5)

---
## 1. The `resource` Block — Syntax and Structure

### Purpose

A `resource` block declares a piece of infrastructure that Terraform should **create, manage, and eventually destroy**. It is the primary building block of every Terraform configuration.

### Syntax

```hcl
resource "<PROVIDER_TYPE>" "<LOCAL_NAME>" {
  argument_name  = value
  another_arg    = value
}
```

| Part | Description |
|------|-------------|
| `resource` | Keyword — tells Terraform this block declares infrastructure |
| `<PROVIDER_TYPE>` | The resource type — always `<provider>_<resourcekind>`, e.g., `aws_instance`, `azurerm_virtual_network`, `google_storage_bucket` |
| `<LOCAL_NAME>` | A name you choose — unique within the resource type in this module |
| Body `{ ... }` | Arguments specific to this resource type (defined by the provider schema) |

### Full Example

```hcl
# main.tf

resource "aws_vpc" "main" {
  cidr_block           = "10.0.0.0/16"
  enable_dns_hostnames = true
  enable_dns_support   = true

  tags = {
    Name        = "main-vpc"
    Environment = "production"
  }
}

resource "aws_subnet" "public" {
  vpc_id                  = aws_vpc.main.id    # Reference to the VPC above
  cidr_block              = "10.0.1.0/24"
  availability_zone       = "us-east-1a"
  map_public_ip_on_launch = true

  tags = {
    Name = "public-subnet"
  }
}

resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
  subnet_id     = aws_subnet.public.id  # Another reference

  tags = {
    Name = "web-server"
  }
}
```

### Provider Type Naming Convention

The resource type always encodes the provider:

| Resource Type | Provider | Resource Kind |
|---------------|----------|---------------|
| `aws_instance` | `aws` | `instance` (EC2) |
| `azurerm_virtual_network` | `azurerm` | `virtual_network` |
| `google_storage_bucket` | `google` | `storage_bucket` |
| `kubernetes_deployment` | `kubernetes` | `deployment` |
| `github_repository` | `github` | `repository` |
| `random_pet` | `random` | `pet` (random name generator) |

> **Exam tip**: The resource type's prefix (`aws_`, `azurerm_`, etc.) tells you which provider manages it. This matters when configuring `required_providers` and when using provider aliases.

---
## 2. Resource Addressing

### Resource Address Format

Every resource in a Terraform configuration has a unique **address** — used for references, `-target` flags, `terraform state show`, and `terraform state mv`.

```
<PROVIDER_TYPE>.<LOCAL_NAME>
```

| Configuration | Address |
|---------------|--------|
| `resource "aws_instance" "web" { ... }` | `aws_instance.web` |
| `resource "aws_vpc" "main" { ... }` | `aws_vpc.main` |
| `resource "azurerm_resource_group" "rg" { ... }` | `azurerm_resource_group.rg` |

### Referencing Attributes

```hcl
# Access an attribute of a managed resource:
# <PROVIDER_TYPE>.<LOCAL_NAME>.<ATTRIBUTE>

resource "aws_vpc" "main" {
  cidr_block = "10.0.0.0/16"
}

resource "aws_subnet" "public" {
  vpc_id     = aws_vpc.main.id        # Reads the 'id' attribute
  cidr_block = "10.0.1.0/24"
}

resource "aws_internet_gateway" "igw" {
  vpc_id = aws_vpc.main.id            # Same reference, different resource
}
```

### Addressing in Modules

```hcl
# Resource inside a module has a prefixed address:
# module.<MODULE_NAME>.<PROVIDER_TYPE>.<LOCAL_NAME>

module "networking" {
  source = "./modules/networking"
}

# Address: module.networking.aws_vpc.main
# CLI usage:
# terraform state show module.networking.aws_vpc.main
# terraform apply -target=module.networking.aws_vpc.main
```

### Addressing with `count`

```hcl
resource "aws_instance" "web" {
  count         = 3
  instance_type = "t3.micro"
  ami           = "ami-0c55b159cbfafe1f0"
}

# Addresses become indexed:
# aws_instance.web[0]
# aws_instance.web[1]
# aws_instance.web[2]

# Reference all IDs as a list:
output "instance_ids" {
  value = aws_instance.web[*].id   # Splat expression
}
```

### Addressing with `for_each`

```hcl
resource "aws_instance" "web" {
  for_each      = { "us-east-1" = "ami-111", "us-west-2" = "ami-222" }
  ami           = each.value
  instance_type = "t3.micro"
}

# Addresses become keyed:
# aws_instance.web["us-east-1"]
# aws_instance.web["us-west-2"]
```

> **Exam tip**: Resource addresses are critical for `-target` operations, `terraform state show`, `terraform state mv`, and `moved` blocks. Know the format: `<type>.<name>`, `<type>.<name>[index]`, and `module.<name>.<type>.<name>`.

---
## 3. Resource Behaviors: CRUD Operations

### The Four Resource Behaviors

| Behavior | When It Happens | Description |
|----------|----------------|-------------|
| **Create** | Resource in config, not in state | Terraform calls the provider API to provision the resource |
| **Read** (Refresh) | Resource in state | Terraform calls provider API to get current attributes and update state |
| **Update** | Resource in config and state, attributes changed | Terraform modifies the resource in-place (mutable attributes) |
| **Destroy** | Resource in state, removed from config | Terraform deletes the resource via provider API |

### Mutable vs Immutable Attributes

Not all attribute changes result in an in-place update. Some attributes are **immutable** — changing them forces the resource to be destroyed and recreated.

```bash
# Mutable attribute change — update in-place (~)
# Changing tags on an EC2 instance:
  ~ resource "aws_instance" "web" {
      ~ tags = {
          ~ "Name" = "old-name" -> "new-name"
        }
    }

# Immutable attribute change — destroy and recreate (-/+)
# Changing the AMI of an EC2 instance:
-/+ resource "aws_instance" "web" {
      -/+ ami = "ami-old" -> "ami-new"  # (forces replacement)
    }
```

### How to Know if an Attribute Is Immutable

The provider documentation marks immutable attributes with **"Forces new resource"** in the attribute description.

```
Argument Reference:
  ami - (Required) AMI to use for the instance.
        Changing this forces a new resource to be created.

  instance_type - (Required) The instance type to use.
                 Changing this forces a new resource to be created.

  tags - (Optional) Map of tags to assign.
         (No 'forces new' — can be changed in-place)
```

### Not All Resource Types Support All CRUD Operations

```hcl
# Some resources are write-once (e.g., certain cryptographic keys):
resource "tls_private_key" "example" {
  algorithm = "RSA"
  rsa_bits  = 4096
  # Changing any attribute forces replacement — no in-place update possible
}

# Some resources cannot be imported or have no meaningful refresh
# (read returns the same config you provided)
```

> **Exam tip**: The plan output symbol `-/+` (destroy then recreate) is triggered by **immutable attribute changes**. Look for `(forces replacement)` in the plan output. Use `lifecycle { create_before_destroy = true }` to reverse the order for zero-downtime replacements.

---
## 4. Required vs Optional Arguments

### Finding Arguments for a Resource

Every resource type has a schema defined by its provider. The schema lists:
- **Required arguments** — must be provided or Terraform will error during `validate`/`plan`
- **Optional arguments** — have a default if omitted
- **Computed attributes** — set by the provider after creation (read-only from config)

### Reading Provider Documentation

```
# Terraform Registry: registry.terraform.io/providers/hashicorp/aws/latest/docs/resources/instance

## Argument Reference

The following arguments are supported:
  ami           - (Required) AMI to use for the instance.
  instance_type - (Required) The instance type to use for the instance.
  subnet_id     - (Optional) The VPC Subnet ID to launch in.
  tags          - (Optional) Map of tags to assign to the resource.
  user_data     - (Optional) User data to provide when launching the instance.

## Attributes Reference (computed, read after creation)
  id            - The instance ID of the created instance.
  public_ip     - The public IP address assigned to the instance (if applicable).
  private_ip    - The private IP address assigned to the instance.
  arn           - The ARN of the instance.
```

### Example — Required and Optional Arguments

```hcl
resource "aws_instance" "web" {
  # Required — must provide these:
  ami           = "ami-0c55b159cbfafe1f0"   # Required
  instance_type = "t3.micro"                 # Required

  # Optional — provider has defaults if omitted:
  monitoring    = false                      # Optional, default: false
  ebs_optimized = false                      # Optional, default: false

  # Computed — set by AWS after creation, cannot be set in config:
  # id         = (known after apply)
  # public_ip  = (known after apply)
  # private_ip = (known after apply)
  # arn        = (known after apply)
}
```

### Using Computed Attributes in Other Resources

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
}

resource "aws_eip" "web_ip" {
  # instance.id is computed — Terraform knows it won't exist until
  # aws_instance.web is created, so it creates the EIP after the instance
  instance = aws_instance.web.id
  domain   = "vpc"
}
```

> **Exam tip**: Computed attributes show as `(known after apply)` in `terraform plan`. They cannot be set in configuration — they are outputs from the provider API after resource creation.

---
## 5. The `data` Block — Reading Existing Infrastructure

### Purpose

A `data` block reads information from **existing infrastructure** (or external sources) that is **not managed by this Terraform workspace**. Data sources are read-only — they never create, update, or destroy anything.

### Syntax

```hcl
data "<PROVIDER_TYPE>" "<LOCAL_NAME>" {
  # Filter arguments to identify which resource to look up
  argument = value
}
```

Referenced as: `data.<PROVIDER_TYPE>.<LOCAL_NAME>.<ATTRIBUTE>`

### Common Use Cases

```hcl
# 1. Look up an existing VPC by tag — not managed by this workspace
data "aws_vpc" "shared" {
  tags = {
    Name = "shared-services-vpc"
  }
}

# Use the looked-up VPC ID in a subnet resource
resource "aws_subnet" "app" {
  vpc_id     = data.aws_vpc.shared.id   # data reference
  cidr_block = "10.100.5.0/24"
}
```

```hcl
# 2. Get the latest Amazon Linux 2023 AMI — automatically updated
data "aws_ami" "amazon_linux" {
  most_recent = true
  owners      = ["amazon"]

  filter {
    name   = "name"
    values = ["al2023-ami-*-x86_64"]
  }

  filter {
    name   = "state"
    values = ["available"]
  }
}

resource "aws_instance" "web" {
  ami           = data.aws_ami.amazon_linux.id  # Always uses latest AMI
  instance_type = "t3.micro"
}
```

```hcl
# 3. Read a secret from AWS Secrets Manager
data "aws_secretsmanager_secret_version" "db_password" {
  secret_id = "production/database/password"
}

resource "aws_db_instance" "main" {
  engine         = "postgres"
  instance_class = "db.t3.medium"
  password       = data.aws_secretsmanager_secret_version.db_password.secret_string
  # Password fetched at plan/apply time — not hardcoded in config
}
```

```hcl
# 4. Get current AWS account ID and region
data "aws_caller_identity" "current" {}
data "aws_region" "current" {}

resource "aws_s3_bucket" "logs" {
  # Construct bucket name using account ID and region — unique globally
  bucket = "logs-${data.aws_caller_identity.current.account_id}-${data.aws_region.current.name}"
}
```

```hcl
# 5. Read outputs from another Terraform workspace's state
data "terraform_remote_state" "networking" {
  backend = "s3"
  config = {
    bucket = "company-terraform-state"
    key    = "production/networking/terraform.tfstate"
    region = "us-east-1"
  }
}

# Access outputs from the networking workspace
resource "aws_eks_cluster" "main" {
  vpc_config {
    subnet_ids = data.terraform_remote_state.networking.outputs.private_subnet_ids
  }
}
```

---
## 6. Resource vs Data Source Comparison

### Side-by-Side Comparison

| Aspect | `resource` Block | `data` Block |
|--------|-----------------|-------------|
| **Keyword** | `resource` | `data` |
| **Purpose** | Declares infrastructure to manage | Reads existing infrastructure |
| **Creates infrastructure?** | Yes | **No** |
| **Destroys infrastructure?** | Yes (`terraform destroy`) | **No** |
| **Appears in state?** | Yes — fully tracked | Partially (cached results) |
| **Reference syntax** | `aws_vpc.main.id` | `data.aws_vpc.shared.id` |
| **Appears in plan `+`/`-`?** | Yes | Shown as `<=` (read) |
| **Lifecycle meta-args?** | Yes (`create_before_destroy`, etc.) | No |
| **Typical use** | Deploying new resources | Looking up IDs, AMIs, secrets |

### Plan Output for Data Sources

```bash
$ terraform plan

  # data.aws_ami.amazon_linux will be read during apply
 <= data "aws_ami" "amazon_linux" {
      + id            = (known after apply)
      + most_recent   = true
      + owners        = ["amazon"]
    }

  # aws_instance.web will be created
  + resource "aws_instance" "web" {
      + ami           = (known after apply)  # depends on data source
      + instance_type = "t3.micro"
    }

Plan: 1 to add, 0 to change, 0 to destroy.
```

### When to Use `data` vs `resource`

```hcl
# Use resource when Terraform SHOULD OWN the lifecycle:
resource "aws_vpc" "new" {
  cidr_block = "10.0.0.0/16"
  # Terraform creates this VPC and is responsible for it
}

# Use data when the resource EXISTS and Terraform should READ it:
data "aws_vpc" "existing" {
  tags = { Name = "shared-services" }
  # This VPC was created manually or by another workspace
  # Terraform must not touch it
}
```

> **Exam tip**: If a resource is managed by **this** workspace → use `resource`. If it was created elsewhere and you just need its attributes → use `data`. Data sources are **never** destroyed by `terraform destroy`.

---
## 7. When Data Sources Read: Plan vs Apply Timing

### Read During `terraform plan` (Most Common)

Most data sources query their APIs during `terraform plan`, so the values are available to the plan:

```hcl
# This data source reads during plan — its result is known
data "aws_availability_zones" "available" {
  state = "available"
}

resource "aws_subnet" "public" {
  count             = 3
  availability_zone = data.aws_availability_zones.available.names[count.index]
  # AZ names are known at plan time, so Terraform can show the full plan
}
```

### Read During `terraform apply` (Deferred)

A data source is deferred to apply time when it depends on a resource that doesn't exist yet:

```hcl
resource "aws_vpc" "main" {
  cidr_block = "10.0.0.0/16"
  # This VPC does not exist yet at plan time
}

# This data source depends on the VPC above — must wait until apply
data "aws_vpc" "main_lookup" {
  id = aws_vpc.main.id           # Depends on a resource being created
  # Cannot read this during plan — aws_vpc.main doesn't exist yet
  # Will read during apply, after the VPC is created
}

# Plan output:
# <= data.aws_vpc.main_lookup will be read during apply
# (depends on resource created during this plan)
```

### Practical Impact

| Scenario | Data source read timing |
|----------|------------------------|
| Querying an existing resource with static filters | Plan time |
| Querying a resource that depends on `resource` being created | Apply time |
| `data.terraform_remote_state` | Plan time (reads from S3/remote) |
| `data.aws_ami` with static filters | Plan time |

> **Exam tip**: When a data source shows `(known after apply)` in the plan, it means Terraform will read the data source **during apply**, not plan. This happens when the data source depends on a `resource` block that hasn't been created yet.

---
## 8. Lifecycle Meta-Arguments

### What Are Meta-Arguments?

Meta-arguments are special arguments available in **every** resource block (they are not provider-specific). They control how Terraform manages the resource lifecycle.

```hcl
resource "<type>" "<name>" {
  # Provider-specific arguments here...

  lifecycle {
    # Meta-arguments inside lifecycle block
  }
}
```

### `create_before_destroy`

By default, when a resource must be replaced (immutable attribute change), Terraform **destroys first, then creates**. Setting `create_before_destroy = true` **reverses this** — create the replacement first, then destroy the old one.

```hcl
resource "aws_launch_template" "app" {
  name_prefix   = "app-"
  image_id      = var.ami_id
  instance_type = "t3.micro"

  lifecycle {
    create_before_destroy = true
    # When AMI changes:
    # 1. New launch template created with new AMI
    # 2. Old launch template destroyed
    # This avoids a gap where no launch template exists
  }
}

# Also essential for resources referenced by other resources:
resource "aws_acm_certificate" "ssl" {
  domain_name       = "example.com"
  validation_method = "DNS"

  lifecycle {
    create_before_destroy = true
    # New cert must exist before the old one is detached from the load balancer
  }
}
```

### `prevent_destroy`

Prevents Terraform from destroying the resource. Any plan that would destroy the resource results in an error — even `terraform destroy`.

```hcl
resource "aws_db_instance" "production" {
  engine         = "postgres"
  engine_version = "15.3"
  instance_class = "db.r6g.large"
  # ... other config ...

  lifecycle {
    prevent_destroy = true
    # Error message when trying to destroy:
    # Error: Instance cannot be destroyed
    # Resource aws_db_instance.production has lifecycle.prevent_destroy
    # set to true. To allow this object to be destroyed, remove or change
    # the prevent_destroy setting on this resource.
  }
}

# To destroy: must first change prevent_destroy = false, then terraform destroy
```

### `ignore_changes`

Tells Terraform to ignore changes to specific attributes when computing diffs. Useful when an attribute is managed outside Terraform (auto-scaling, manual tags, etc.).

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  lifecycle {
    ignore_changes = [
      tags,          # Tags managed by the tagging team — Terraform should not touch
      user_data,     # User data changed by automation post-launch
    ]
  }
}

# Ignore ALL changes (use with extreme caution — Terraform essentially stops managing this resource)
lifecycle {
  ignore_changes = all
}
```

### `replace_triggered_by`

Force the resource to be replaced when another resource or resource attribute changes — even if this resource's own attributes haven't changed.

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  lifecycle {
    replace_triggered_by = [
      aws_launch_template.app.id
      # When the launch template changes, this instance is also replaced
    ]
  }
}
```

### Lifecycle Meta-Argument Summary

| Meta-Argument | Purpose | Key Use Case |
|--------------|---------|-------------|
| `create_before_destroy` | Reverse replace order | Zero-downtime certificate/LT rotation |
| `prevent_destroy` | Block any destroy | Protect production databases |
| `ignore_changes` | Skip diff on attributes | Externally managed tags, auto-scaling |
| `replace_triggered_by` | Force replace when dependency changes | Rolling instance updates |

> **Exam tip**: `prevent_destroy = true` does NOT prevent replacement (`-/+`) — it only prevents pure destruction (`-`). A replacement plan (destroy + create) will also be blocked because the destroy step is part of the plan.

---
## 9. Complete Examples

### Example 1: VPC with Data Source Lookup

```hcl
# Get the current AWS region
data "aws_region" "current" {}

# Look up an existing shared VPC (not managed here)
data "aws_vpc" "shared" {
  filter {
    name   = "tag:Name"
    values = ["shared-services"]
  }
}

# Get latest Amazon Linux 2023 AMI
data "aws_ami" "al2023" {
  most_recent = true
  owners      = ["amazon"]
  filter {
    name   = "name"
    values = ["al2023-ami-*-x86_64"]
  }
}

# Create a security group in the shared VPC
resource "aws_security_group" "app" {
  name   = "app-sg"
  vpc_id = data.aws_vpc.shared.id   # data reference

  ingress {
    from_port   = 8080
    to_port     = 8080
    protocol    = "tcp"
    cidr_blocks = [data.aws_vpc.shared.cidr_block]  # another data reference
  }
}

# Create an EC2 instance using the latest AMI
resource "aws_instance" "app" {
  ami                    = data.aws_ami.al2023.id   # data reference
  instance_type          = "t3.micro"
  subnet_id              = data.aws_vpc.shared.id   # illustrative
  vpc_security_group_ids = [aws_security_group.app.id]  # resource reference

  tags = {
    Name   = "app-server"
    Region = data.aws_region.current.name
  }
}
```

### Example 2: Database with Full Lifecycle Protection

```hcl
# Read database password from Secrets Manager (not hardcoded)
data "aws_secretsmanager_secret_version" "db_creds" {
  secret_id = "production/rds/master"
}

# Parse the secret as JSON
locals {
  db_creds = jsondecode(data.aws_secretsmanager_secret_version.db_creds.secret_string)
}

resource "aws_db_instance" "production" {
  identifier     = "prod-database"
  engine         = "postgres"
  engine_version = "15.4"
  instance_class = "db.r6g.large"
  allocated_storage = 100

  username = local.db_creds.username
  password = local.db_creds.password

  lifecycle {
    prevent_destroy = true          # Protect against accidental deletion
    ignore_changes  = [password]    # Password rotated externally via Secrets Manager
  }
}
```

### Example 3: Load Balancer Certificate with create_before_destroy

```hcl
resource "aws_acm_certificate" "api" {
  domain_name       = "api.example.com"
  validation_method = "DNS"

  lifecycle {
    # New cert must be validated and ready BEFORE old cert is removed from LB
    create_before_destroy = true
  }
}

resource "aws_lb_listener" "https" {
  load_balancer_arn = aws_lb.main.arn
  port              = 443
  protocol          = "HTTPS"
  certificate_arn   = aws_acm_certificate.api.arn

  default_action {
    type             = "forward"
    target_group_arn = aws_lb_target_group.app.arn
  }
}
```

---
## 10. Exam-Style Practice Questions

---

**Q1: A Terraform configuration includes the following data source:**

```hcl
data "aws_vpc" "prod" {
  tags = { Environment = "production" }
}
```

**What will happen when `terraform destroy` is run on a workspace containing this data source?**

A) The VPC identified by the data source will be deleted  
B) The data source will be removed from state, but the actual VPC will not be deleted  
C) `terraform destroy` will skip the data source and only destroy resources  
D) Terraform will error because data sources cannot be referenced when destroying  

<details>
<summary>Answer</summary>

**Answer: C (or B depending on interpretation)**  
Data sources are **read-only** — `terraform destroy` only destroys resources managed by `resource` blocks. The VPC looked up via the data source is not touched. Data sources do not have a destroy operation. Terraform simply removes the cached data source result from state — the actual infrastructure is unaffected. The key rule is: data sources never create or destroy infrastructure.

</details>

---

**Q2: A developer adds `lifecycle { prevent_destroy = true }` to a production database resource. A teammate later runs `terraform apply` with a change that would cause the database to be replaced (immutable attribute change). What happens?**

A) The database is destroyed and recreated — `prevent_destroy` only blocks `terraform destroy` command  
B) Terraform errors during plan: the plan includes a destroy step as part of replacement, which `prevent_destroy` blocks  
C) Terraform proceeds with the replacement but asks for an additional confirmation  
D) The immutable attribute change is silently ignored because `prevent_destroy` is set  

<details>
<summary>Answer</summary>

**Answer: B**  
`prevent_destroy = true` blocks **any plan that includes a destroy step for that resource** — including the destroy step in a `-/+` replacement operation. Terraform errors at plan time with `Error: Instance cannot be destroyed`. The teammate must first remove `prevent_destroy`, plan the replacement, and manually confirm the risk. Option A is incorrect — `prevent_destroy` blocks any destroy, not just `terraform destroy` command-initiated ones.

</details>

---

**Q3: A configuration references `data.aws_ami.latest.id` in an `aws_instance` resource. The `aws_ami` data source uses `most_recent = true`. What is the risk of this pattern, and how can it be mitigated?**

A) The AMI ID is always known at plan time, so there is no risk  
B) A new AMI release between plan and apply could cause the instance to be created with a different AMI than what was shown in the plan  
C) Data sources cannot use `most_recent = true` — this will cause a validation error  
D) The instance will always be replaced on every `terraform apply` because the AMI changes daily  

<details>
<summary>Answer</summary>

**Answer: B**  
The `aws_ami` data source with `most_recent = true` is evaluated at plan time. If a newer AMI is released between `terraform plan` and `terraform apply`, the apply will use the newly-latest AMI — which may differ from the plan. Mitigation: use `terraform plan -out=plan.tfplan` and `terraform apply plan.tfplan` to apply exactly the saved plan. Another option is to pin the AMI ID to a specific version using `lifecycle { ignore_changes = [ami] }` after initial creation.

</details>

---
## 11. Real-World Scenarios

<details>
<summary>Scenario 1 — Zero-Downtime SSL Certificate Rotation</summary>

**Situation:**
A company's wildcard SSL certificate for `*.example.com` expires annually. When the Terraform config is updated with a new certificate, the load balancer listener references the certificate ARN. Without `create_before_destroy`, Terraform would destroy the old certificate first, briefly leaving the HTTPS listener with no valid certificate and causing a service outage.

**Configuration:**

```hcl
resource "aws_acm_certificate" "wildcard" {
  domain_name               = "*.example.com"
  subject_alternative_names = ["example.com"]
  validation_method         = "DNS"

  lifecycle {
    create_before_destroy = true
    # Ensures new certificate is fully validated and available
    # BEFORE the old certificate resource is destroyed
  }

  tags = { Purpose = "wildcard-ssl" }
}

# Certificate validation (DNS record proves ownership)
resource "aws_route53_record" "cert_validation" {
  for_each = {
    for dvo in aws_acm_certificate.wildcard.domain_validation_options : dvo.domain_name => {
      name  = dvo.resource_record_name
      type  = dvo.resource_record_type
      value = dvo.resource_record_value
    }
  }

  zone_id = data.aws_route53_zone.main.zone_id
  name    = each.value.name
  type    = each.value.type
  records = [each.value.value]
  ttl     = 60

  lifecycle {
    create_before_destroy = true  # Propagate to validation records too
  }
}

resource "aws_acm_certificate_validation" "wildcard" {
  certificate_arn         = aws_acm_certificate.wildcard.arn
  validation_record_fqdns = [for r in aws_route53_record.cert_validation : r.fqdn]
}

resource "aws_lb_listener" "https" {
  load_balancer_arn = aws_lb.main.arn
  port              = 443
  protocol          = "HTTPS"
  certificate_arn   = aws_acm_certificate_validation.wildcard.certificate_arn
}
```

**Execution:**

```bash
# Certificate domain/SANs updated in config
terraform plan
# Shows: create new cert first, then destroy old cert
# create_before_destroy ensures no gap in HTTPS service

terraform apply plan.tfplan
# 1. New certificate created and validated (takes ~1 minute)
# 2. Listener updated to point to new certificate
# 3. Old certificate destroyed
# Zero HTTPS downtime throughout
```

**Expected outcome:**
- No SSL downtime during certificate rotation
- New certificate is validated before the old one is removed
- Safe to automate in CI/CD

**Exam sub-objective demonstrated:** `create_before_destroy` lifecycle meta-argument, resource dependencies, data source usage.

</details>

<details>
<summary>Scenario 2 — Protecting a Production Database from Accidental Destruction</summary>

**Situation:**
A junior engineer accidentally removes the `aws_db_instance.production` resource block from the Terraform configuration while refactoring. Without protection, the next `terraform apply` would permanently destroy the production database. The team wants a safety net.

**Configuration with protection:**

```hcl
resource "aws_db_instance" "production" {
  identifier            = "prod-postgres"
  engine                = "postgres"
  engine_version        = "15.4"
  instance_class        = "db.r6g.xlarge"
  allocated_storage     = 500
  storage_encrypted     = true
  deletion_protection   = true  # AWS-level protection as well

  lifecycle {
    prevent_destroy = true    # Terraform-level protection
    ignore_changes  = [
      password,               # Rotated by AWS Secrets Manager rotation
      snapshot_identifier,    # Set by restore operations, not Terraform
    ]
  }
}
```

**What happens when the resource block is accidentally deleted:**

```bash
$ terraform plan
╷
│ Error: Instance cannot be destroyed
│
│   on main.tf line 1:
│   1: resource "aws_db_instance" "production" {
│
│ Resource aws_db_instance.production has lifecycle.prevent_destroy set
│ to true. To allow this object to be destroyed, remove or change the
│ prevent_destroy setting on this resource.
╵
# The plan FAILS before any change is made — database is safe
```

**Deliberate decommission process:**

```bash
# Step 1: Take a final snapshot
# (done outside Terraform or via a one-off resource)

# Step 2: Remove prevent_destroy in config
# lifecycle { prevent_destroy = false }

# Step 3: Also disable AWS deletion protection
# deletion_protection = false

# Step 4: Commit change through PR review process (not a hot-fix)

# Step 5: Apply with explicit confirmation
terraform plan   # Shows the destroy
terraform apply  # Requires typing 'yes'
```

**Expected outcome:**
- Accidental removal from config is caught at plan time — no data loss
- Deliberate destruction requires explicit config change + PR review + manual confirmation
- Two layers of protection: Terraform `prevent_destroy` and AWS `deletion_protection`

**Exam sub-objective demonstrated:** `prevent_destroy` lifecycle, `ignore_changes`, defense in depth for critical resources.

</details>

<details>
<summary>Scenario 3 — Using a Data Source to Always Deploy the Latest AMI</summary>

**Situation:**
A team manages EC2 instances that should always use the latest security-patched Amazon Linux 2023 AMI. Hardcoding AMI IDs means the config goes stale as new AMIs are released. The team wants Terraform to automatically use the latest AMI each time it runs.

**Configuration:**

```hcl
# data source always resolves to the current latest AL2023 AMI
data "aws_ami" "al2023" {
  most_recent = true
  owners      = ["amazon"]

  filter {
    name   = "name"
    values = ["al2023-ami-*-x86_64"]
  }

  filter {
    name   = "virtualization-type"
    values = ["hvm"]
  }

  filter {
    name   = "state"
    values = ["available"]
  }
}

resource "aws_launch_template" "app" {
  name_prefix   = "app-"
  image_id      = data.aws_ami.al2023.id  # Always latest AMI
  instance_type = "t3.medium"

  lifecycle {
    create_before_destroy = true
    # When AMI changes, new LT created before old is destroyed
  }
}

resource "aws_autoscaling_group" "app" {
  name                = "app-asg"
  min_size            = 2
  max_size            = 10
  desired_capacity    = 2
  vpc_zone_identifier = var.subnet_ids

  launch_template {
    id      = aws_launch_template.app.id
    version = "$Latest"
  }

  instance_refresh {
    strategy = "Rolling"  # Rolling update — no downtime
    preferences {
      min_healthy_percentage = 90
    }
  }
}
```

**Operations:**

```bash
# Monthly security update run:
terraform plan
# If a new AMI was released, plan shows:
# -/+ aws_launch_template.app (forces replacement — AMI changed)
# ~ aws_autoscaling_group.app (update to reference new LT version)

terraform apply
# 1. New launch template created with new AMI (create_before_destroy)
# 2. ASG updated to use new LT version
# 3. Instance refresh rolls out new AMI to EC2 instances (rolling)
# 4. Old launch template destroyed

# If no new AMI was released:
# Plan: 0 to add, 0 to change, 0 to destroy
# (data source still resolves, but same AMI ID — no diff)
```

**Expected outcome:**
- EC2 instances always run the latest patched AMI
- AMI updates are zero-downtime via rolling instance refresh
- No manual AMI ID management required

**Exam sub-objective demonstrated:** `data` block for dynamic lookups, `create_before_destroy` lifecycle, resource vs data source distinction.

</details>

<details>
<summary>Scenario 4 — Using `ignore_changes` for Auto-Scaled Resources</summary>

**Situation:**
A Kubernetes cluster's node group has `desired_size = 3` in Terraform. The cluster's Cluster Autoscaler changes the desired node count dynamically based on load. Without `ignore_changes`, every `terraform plan` would show a diff wanting to reset `desired_size` back to 3, undoing the autoscaler's work.

**Problem without `ignore_changes`:**

```bash
# Autoscaler has scaled the group to 7 nodes
$ terraform plan
  ~ resource "aws_eks_node_group" "workers" {
      ~ scaling_config {
          ~ desired_size = 7 -> 3   # WRONG: Terraform wants to undo autoscaling!
        }
    }
# Applying this would disrupt the autoscaler and potentially the cluster
```

**Solution with `ignore_changes`:**

```hcl
resource "aws_eks_node_group" "workers" {
  cluster_name    = aws_eks_cluster.main.name
  node_group_name = "workers"
  node_role_arn   = aws_iam_role.node.arn
  subnet_ids      = var.private_subnet_ids
  instance_types  = ["m5.large"]

  scaling_config {
    desired_size = 3   # Initial size — managed by autoscaler after creation
    min_size     = 2
    max_size     = 20
  }

  lifecycle {
    ignore_changes = [
      scaling_config[0].desired_size,
      # Terraform manages min/max but ignores desired_size changes
      # Cluster Autoscaler owns desired_size after initial creation
    ]
  }
}
```

**After fix:**

```bash
# Autoscaler has scaled to 7 nodes
$ terraform plan
# aws_eks_node_group.workers: Refreshing state...
No changes. Your infrastructure matches the configuration.
# Terraform sees desired_size = 7 in state but ignores the diff
# min_size and max_size changes would still be applied
```

**Expected outcome:**
- Cluster Autoscaler can scale the node group up and down freely
- Terraform does not reset desired_size on each plan/apply
- min_size and max_size are still Terraform-managed

**Exam sub-objective demonstrated:** `ignore_changes` lifecycle meta-argument, collaboration between Terraform and external automation.

</details>

<details>
<summary>Scenario 5 — Reading Cross-Workspace Outputs with `terraform_remote_state`</summary>

**Situation:**
A company uses three Terraform workspaces: `networking`, `security`, and `application`. The application workspace needs the VPC ID, subnet IDs, and security group IDs created by the networking and security workspaces. Cross-workspace communication is handled via `terraform_remote_state` data sources.

**networking workspace outputs.tf:**

```hcl
output "vpc_id" {
  value       = aws_vpc.main.id
  description = "Main VPC ID"
}

output "private_subnet_ids" {
  value       = [for s in aws_subnet.private : s.id]
  description = "Private subnet IDs in all AZs"
}
```

**security workspace outputs.tf:**

```hcl
output "app_security_group_id" {
  value       = aws_security_group.app.id
  description = "Security group for application servers"
}
```

**application workspace main.tf:**

```hcl
# Read outputs from networking workspace
data "terraform_remote_state" "networking" {
  backend = "s3"
  config = {
    bucket = "company-terraform-state"
    key    = "production/networking/terraform.tfstate"
    region = "us-east-1"
  }
}

# Read outputs from security workspace
data "terraform_remote_state" "security" {
  backend = "s3"
  config = {
    bucket = "company-terraform-state"
    key    = "production/security/terraform.tfstate"
    region = "us-east-1"
  }
}

# Use the cross-workspace values
resource "aws_instance" "app" {
  count         = 3
  ami           = data.aws_ami.al2023.id
  instance_type = "m5.large"

  # From networking workspace
  subnet_id     = data.terraform_remote_state.networking.outputs.private_subnet_ids[count.index % length(data.terraform_remote_state.networking.outputs.private_subnet_ids)]

  # From security workspace
  vpc_security_group_ids = [
    data.terraform_remote_state.security.outputs.app_security_group_id
  ]

  tags = {
    # From networking workspace
    VPC = data.terraform_remote_state.networking.outputs.vpc_id
  }
}
```

**Expected outcome:**
- Application workspace deploys EC2 instances into the correct subnets and security groups
- No hard-coded IDs — all values are dynamically read from other workspaces' state
- Networking or security changes are automatically picked up next time application workspace plans
- Clean separation of concerns: each workspace manages its own domain

**Exam sub-objective demonstrated:** `data` block with `terraform_remote_state`, cross-workspace data sharing, data source vs resource distinction.

</details>

---
## Quick-Reference Cheat Sheet

```
Resource and Data Blocks Cheat Sheet
═══════════════════════════════════════════════════════════════════

RESOURCE BLOCK
  resource "<type>" "<name>" { ... }
  Address:    <type>.<name>
  Attribute:  <type>.<name>.<attr>
  With count: <type>.<name>[index]  /  <type>.<name>[*].attr (splat)
  With f/e:   <type>.<name>["key"]
  In module:  module.<mod>.<type>.<name>

DATA BLOCK
  data "<type>" "<name>" { ... }
  Address:    data.<type>.<name>
  Attribute:  data.<type>.<name>.<attr>
  READ-ONLY:  never creates, updates, or destroys infrastructure
  Not affected by terraform destroy

LIFECYCLE META-ARGUMENTS
  create_before_destroy = true
    → Create replacement BEFORE destroying original
    → Use for: certs, launch templates, any resource another depends on

  prevent_destroy = true
    → Block ANY plan that includes destroying this resource
    → Includes -/+ replacement plans (they have a destroy step)
    → Use for: production databases, critical stateful resources

  ignore_changes = [attr1, attr2]  OR  ignore_changes = all
    → Skip diff on listed attributes
    → Use when: external automation manages the attribute
    → Examples: autoscaler desired_size, externally-rotated passwords

  replace_triggered_by = [resource.name]
    → Force replace when referenced resource/attribute changes

PLAN SYMBOLS FOR DATA SOURCES
  <=  Read (data source fetch — no infrastructure change)

KEY RULES
  data  → reads existing  → not destroyed by terraform destroy
  resource → manages lifecycle → destroyed by terraform destroy
  prevent_destroy blocks replacements too (not just explicit destroy)
  create_before_destroy: create new, then destroy old (reverse default)
  ignore_changes: Terraform won't drift-detect those attributes
═══════════════════════════════════════════════════════════════════
```